In [7]:
# Generador de datasets realistas y correlacionados para Shoply (usuarios & compras)
# - Crea dos CSVs: usuarios.csv y compras.csv
# - Mantiene relaciones plausibles entre ingreso, canal, plan, tickets, tiempo de resolución, churn y actividad de compra.
# - Incluye estacionalidad y efectos por país.
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

rng = np.random.default_rng(42)
np.random.seed(42)
random.seed(42)

# -----------------------------
# Parámetros
# -----------------------------
N_USERS = 1200
START_DATE = pd.Timestamp("2024-01-01")
END_DATE   = pd.Timestamp("2025-09-30")

countries = ["Mexico","Chile","Colombia","Argentina","Peru"]
country_p = [0.35, 0.18, 0.2, 0.18, 0.09]

channels = ["Ads","SEO","Referral","Direct"]
plans    = ["Free","Basic","Premium","Pro"]

# Medianas de ingreso por país (USD/mes) + dispersión relativa
country_income_med = {
    "Mexico":   1400,
    "Chile":    1600,
    "Colombia": 1200,
    "Argentina":1100,
    "Peru":     1000,
}
country_income_sigma = 0.5  # lognormal sigma (dispersion)

# Precios y ticket size base por plan
plan_price_anchor = {
    "Free":    0.0,
    "Basic":   12.0,
    "Premium": 24.0,
    "Pro":     39.0,
}
# Ratio multiplicador de compra por plan (impacta monto y frecuencia)
plan_spend_mult = {
    "Free":    0.3,
    "Basic":   0.9,
    "Premium": 1.15,
    "Pro":     1.35,
}

# Efectos por canal sobre propensión a pagar y churn (aprox.)
channel_pay_lift = {"Referral": 1.15, "SEO": 1.05, "Direct": 1.00, "Ads": 0.85}
channel_churn_lift = {"Referral": 0.85, "SEO": 0.95, "Direct": 1.00, "Ads": 1.10}

# Estacionalidad de compra por mes (factor multiplicativo)
seasonality = {
    1: 0.95,  2: 0.90,  3: 1.00,
    4: 1.05,  5: 1.00,  6: 0.95,
    7: 0.90,  8: 0.95,  9: 1.05,
    10:1.10, 11:1.20, 12:1.30
}

# -----------------------------
# Helper functions
# -----------------------------
def draw_income(country, size=1):
    """Lognormal alrededor de la mediana país."""
    med = country_income_med[country]
    # Convertir mediana a mu para lognormal: median = exp(mu) -> mu = ln(median)
    mu = np.log(med)
    sigma = country_income_sigma
    return np.random.lognormal(mean=mu, sigma=sigma, size=size)

def choose_plan(income, channel):
    """Probabilidades de plan según ingreso y canal."""
    # limites en USD/mes, heurísticos
    if income < 800:
        base = [0.65, 0.25, 0.09, 0.01]   # Free, Basic, Premium, Pro
    elif income < 1500:
        base = [0.40, 0.35, 0.20, 0.05]
    elif income < 3000:
        base = [0.25, 0.30, 0.32, 0.13]
    else:
        base = [0.15, 0.25, 0.35, 0.25]
    # canal modifica ligeramente: Referral/SEO reducen Free y suben Premium/Pro
    ch_lifts = {
        "Referral": [0.9, 0.95, 1.1, 1.15],
        "SEO":      [0.95, 1.0, 1.05, 1.05],
        "Direct":   [1.0, 1.0, 1.0, 1.0],
        "Ads":      [1.05, 1.05, 0.95, 0.9],
    }
    adj = np.array(base) * np.array(ch_lifts[channel])
    adj = adj / adj.sum()
    return np.random.choice(plans, p=adj)

def poisson_with_floor(lmbd):
    """Poisson con lbd >= 0 pero devolviendo 0 si lbd ~ 0."""
    if lmbd <= 0:
        return 0
    return np.random.poisson(lmbd)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# -----------------------------
# Generar usuarios
# -----------------------------
user_id = np.arange(1, N_USERS+1)
country = rng.choice(countries, size=N_USERS, p=country_p)
age = rng.integers(18, 66, size=N_USERS)
gender = rng.choice(["M","F"], size=N_USERS, p=[0.5,0.5])

# signup_channel depende levemente de país (ads más alto en MX/CO, referral alto en Chile/AR)
signup_channel = []
for c in country:
    if c in ["Mexico","Colombia"]:
        p = [0.40, 0.23, 0.18, 0.19]  # Ads, SEO, Referral, Direct
    elif c in ["Chile","Argentina"]:
        p = [0.28, 0.27, 0.25, 0.20]
    else:  # Peru
        p = [0.35, 0.23, 0.18, 0.24]
    signup_channel.append(np.random.choice(channels, p=p))
signup_channel = np.array(signup_channel)

# Ingreso mensual por país
monthly_income = np.concatenate([draw_income(c, 1) for c in country])

# Plan por usuario
subscription_type = np.array([choose_plan(inc, ch) for inc, ch in zip(monthly_income, signup_channel)])

# Tickets de soporte y tiempo de resolución (correlacionados con plan e ingreso)
base_ticket_rate = {
    "Free":  1.6,
    "Basic": 1.3,
    "Premium":1.0,
    "Pro":   0.9
}
support_tickets = []
for plan, inc in zip(subscription_type, monthly_income):
    # Menor ingreso -> +tickets; cada 1000 USD por debajo de 1500 agrega 0.2 tickets esperados
    inc_adj = max(0, (1500 - inc) / 1000.0) * 0.2
    lam = max(0.05, base_ticket_rate[plan] + inc_adj)
    support_tickets.append(poisson_with_floor(lam))
support_tickets = np.array(support_tickets)

# resolution_time correlaciona + con tickets y es menor en Premium/Pro (SLA mejor)
plan_rt_adj = {"Free": 1.15, "Basic": 1.05, "Premium": 0.95, "Pro": 0.9}
resolution_time = np.abs(
    np.random.normal(8.5, 2.8, N_USERS) * np.vectorize(plan_rt_adj.get)(subscription_type) + support_tickets*0.6
)

# Meses activos (si churn=1 será acotado después)
active_months = rng.integers(1, 37, size=N_USERS)  # 1..36

# Probabilidad de churn basada en tickets, tiempo de resolución, plan, canal e ingreso
# logit ~ +tickets + rt + (Free/Basic) - ingreso + efecto canal
plan_churn_base = {"Free": 0.55, "Basic": 0.38, "Premium": 0.22, "Pro": 0.15}
logit = (
    -2.0
    + 0.28*support_tickets
    + 0.10*resolution_time
    + np.array([plan_churn_base[p] for p in subscription_type])
    - 0.00025*monthly_income
    + np.array([np.log(channel_churn_lift[c]) for c in signup_channel])
)
p_churn = np.clip(sigmoid(logit), 0.02, 0.95)
churn = rng.binomial(1, p_churn)

# Ajustar active_months si churn=1 (vida más corta)
# Si churn=1, truncar meses activos a un percentil del actual
active_months_adj = active_months.copy()
mask_churn = churn == 1
active_months_adj[mask_churn] = np.maximum(1, (active_months[mask_churn] * rng.uniform(0.2, 0.8, mask_churn.sum())).astype(int))

# Construir DataFrame usuarios
usuarios = pd.DataFrame({
    "user_id": user_id,
    "age": age,
    "gender": gender,
    "country": country,
    "signup_channel": signup_channel,
    "monthly_income": monthly_income.round(2),
    "subscription_type": subscription_type,
    "support_tickets": support_tickets.astype(int),
    "resolution_time": np.round(resolution_time, 2),
    "active_months": active_months_adj.astype(int),
    "churn": churn.astype(int),
})

# Introducir nulos realistas (MCAR pequeño) y algunos outliers controlados
null_idx = usuarios.sample(frac=0.015, random_state=7).index
usuarios.loc[null_idx, "monthly_income"] = np.nan
usuarios.loc[usuarios.sample(frac=0.01, random_state=8).index, "resolution_time"] = np.nan
usuarios.loc[usuarios.sample(frac=0.008, random_state=9).index, "age"] = np.nan

# Outliers: algunos ingresos muy altos y tiempos de resolución exagerados
oi = usuarios.sample(frac=0.008, random_state=10).index
usuarios.loc[oi, "monthly_income"] = usuarios["monthly_income"].median() * 5
oi2 = usuarios.sample(frac=0.006, random_state=11).index
usuarios.loc[oi2, "resolution_time"] = usuarios["resolution_time"].quantile(0.99) * 3

# -----------------------------
# Generar compras
# -----------------------------
# Frecuencia base mensual por plan
plan_freq_month = {"Free": 0.10, "Basic": 0.35, "Premium": 0.55, "Pro": 0.75}

# Monto base por compra según plan (alrededor de price_anchor con varianza)
def draw_purchase_amount(plan, country):
    base = plan_price_anchor[plan] * (0.7 + 0.6*np.random.rand())  # alrededor del precio
    # Efecto país (paridad de poder adquisitivo aproximada)
    country_mult = {
        "Mexico": 1.00, "Chile": 1.10, "Colombia": 0.90, "Argentina": 0.85, "Peru": 0.88
    }[country]
    return max(0.99, np.random.normal(base, base*0.35) * country_mult)

# Para cada usuario, simular número de compras y fechas dentro de su ventana activa
purchase_rows = []
purchase_id = 1

for idx, row in usuarios.iterrows():
    uid = int(row["user_id"])
    plan = row["subscription_type"]
    ctry = row["country"]
    ch   = row["signup_channel"]
    months_active = int(row["active_months"])

    # fecha de inicio aleatoria dentro del rango, pero asegurando que meses activos estén dentro del periodo total
    total_days = (END_DATE - START_DATE).days + 1
    max_start = max(1, total_days - (months_active*30))
    start_offset = rng.integers(0, max_start)
    start_date = START_DATE + timedelta(days=int(start_offset))
    # fecha de término = start + active_months
    end_date = min(start_date + pd.DateOffset(months=months_active), END_DATE)

    # Intensidad mensual afectada por plan, canal, ingreso y churn (si churn=1, recorta actividad a mitad)
    lam_m = plan_freq_month[plan]
    lam_m *= channel_pay_lift[ch]
    inc = row["monthly_income"] if pd.notna(row["monthly_income"]) else np.nanmedian(usuarios["monthly_income"])
    lam_m *= np.clip(0.7 + (inc/3000), 0.6, 1.6)  # más ingreso -> +frecuencia hasta ~1.6x
    if row["churn"] == 1:
        lam_m *= 0.6  # menor frecuencia si churn pronto

    # Número esperado total de compras ~ Poisson(lam_m * months_active)
    expected_purchases = lam_m * months_active
    n_p = poisson_with_floor(expected_purchases)
    if n_p == 0:
        continue

    # Distribuir fechas uniformemente dentro de la ventana, luego ajustar por estacionalidad
    # Proceso: proponer más fechas y luego muestrear por prob. proporcional a seasonality
    days_span = (end_date - start_date).days + 1
    candidate_days = rng.integers(0, max(days_span, 1), size=max(n_p*3, 30))
    candidate_dates = [start_date + timedelta(days=int(d)) for d in candidate_days]
    # Ponderación por estacionalidad del mes
    weights = np.array([seasonality[dt.month] for dt in candidate_dates], dtype=float)
    if weights.sum() == 0:
        weights = np.ones_like(weights)
    weights = weights / weights.sum()
    chosen_idx = rng.choice(len(candidate_dates), size=n_p, replace=False, p=weights)
    chosen_dates = [candidate_dates[i] for i in chosen_idx]

    for dt in chosen_dates:
        amount = draw_purchase_amount(plan, ctry) * plan_spend_mult[plan]
        purchase_rows.append((purchase_id, uid, round(float(amount), 2), pd.Timestamp(dt).date()))
        purchase_id += 1

compras = pd.DataFrame(purchase_rows, columns=["purchase_id","user_id","purchase_amount","purchase_date"])

# Ordenar y resetear purchase_id consecutivo
compras = compras.sort_values("purchase_date").reset_index(drop=True)
compras["purchase_id"] = np.arange(1, len(compras)+1)

# -----------------------------
# Guardar archivos
# -----------------------------
usuarios.to_csv("./data/usuarios_shoply.csv", index=False)
compras.to_csv("./data/compras_shoply.csv", index=False)

# Mostrar pequeñas vistas previas para verificación

# Resumen útil para el profesor
summary = {
    "n_usuarios": [usuarios["user_id"].nunique()],
    "n_compras": [compras["purchase_id"].nunique()],
    "pct_pagadores": [round(compras["user_id"].nunique() / usuarios["user_id"].nunique(), 3)],
    "churn_rate": [round(usuarios["churn"].mean(), 3)],
    "arpu": [round(compras["purchase_amount"].sum() / usuarios["user_id"].nunique(), 2)],
    "arppu": [round(compras["purchase_amount"].sum() / max(compras["user_id"].nunique(), 1), 2)],
}
summary_df = pd.DataFrame(summary)

In [10]:
usuarios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            1200 non-null   int64  
 1   age                1190 non-null   float64
 2   gender             1200 non-null   object 
 3   country            1200 non-null   object 
 4   signup_channel     1200 non-null   object 
 5   monthly_income     1182 non-null   float64
 6   subscription_type  1200 non-null   object 
 7   support_tickets    1200 non-null   int64  
 8   resolution_time    1188 non-null   float64
 9   active_months      1200 non-null   int64  
 10  churn              1200 non-null   int64  
dtypes: float64(3), int64(4), object(4)
memory usage: 103.3+ KB


In [11]:
compras.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6770 entries, 0 to 6769
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   purchase_id      6770 non-null   int64  
 1   user_id          6770 non-null   int64  
 2   purchase_amount  6770 non-null   float64
 3   purchase_date    6770 non-null   object 
dtypes: float64(1), int64(2), object(1)
memory usage: 211.7+ KB
